In [3]:
import numpy as np
import pandas as pd

class Hospital:
    def __init__(self, id, name, capabilities, travel_time, confirm_delay, accept_prob, load, uncertainty):
        self.id = id
        self.name = name
        self.capabilities = set(capabilities) # 병원이 가진 역량 (예: 'CTA', 'thrombectomy')
        
        # 3.4 목적 함수(Objective Function) 변수들
        self.travel_time = travel_time        # T: 예상 이동 시간
        self.confirm_delay = confirm_delay    # C: 승인까지 예상 지연 시간
        self.accept_prob = accept_prob        # A: 수용 확률 (예측치)
        self.load = load                      # L: 현재 병상 부하/혼잡도
        self.uncertainty = uncertainty        # U: 데이터 불확실성 (오래된 데이터일수록 페널티)

class SynapseH_Optimizer:
    def __init__(self, weights):
        # 목적 함수의 가중치 (w1 ~ w5)
        self.weights = weights

    def calculate_cost(self, hospital):
        """
        제안서 3.4의 수학적 목적 함수를 계산합니다.
        Cost = w1*T + w2*C + w3*(1-A) + w4*L + w5*U
        """
        w1, w2, w3, w4, w5 = self.weights
        
        cost = (w1 * hospital.travel_time +
                w2 * hospital.confirm_delay +
                w3 * (1 - hospital.accept_prob) * 100 +  # 확률 역수를 스케일링
                w4 * hospital.load +
                w5 * hospital.uncertainty)
        return cost

    def rank_destinations(self, required_capabilities, hospital_network):
        """
        환자의 요구 역량을 충족하는 병원을 필터링하고, Cost 기준으로 랭킹을 매깁니다.
        """
        eligible_hospitals = []
        
        # 1. Hard Constraints (역량 필터링 - 거리가 아무리 가까워도 역량이 없으면 제외)
        required_set = set(required_capabilities)
        for h in hospital_network:
            if required_set.issubset(h.capabilities):
                eligible_hospitals.append(h)
        
        if not eligible_hospitals:
            return "ESCALATE TO MEDICAL CONTROL: 조건에 맞는 병원이 없습니다." # Abstention Rule
            
        # 2. 최적화 비용 계산 및 정렬 (Ranking)
        ranked_list = []
        for h in eligible_hospitals:
            cost = self.calculate_cost(h)
            ranked_list.append({'Hospital': h.name, 'Cost_Score': round(cost, 2), 'Accept_Prob': f"{h.accept_prob*100}%", 'ETA': f"{h.travel_time}m"})
            
        # Cost가 낮을수록 좋음 (오름차순 정렬)
        ranked_list = sorted(ranked_list, key=lambda x: x['Cost_Score'])
        
        # UI에 보여줄 Top 3 반환
        return pd.DataFrame(ranked_list[:3])

# ==========================================
# 🚀 베타 테스트 실행 (Synthetic Data)
# ==========================================

# 1. 모델 가중치 세팅 (Travel, Delay, Risk, Load, Uncertainty)
weights = (1.0, 1.2, 0.8, 0.5, 1.5)
synapse_engine = SynapseH_Optimizer(weights)

# 2. 가상의 지역 병원 네트워크 데이터 생성 (Digital Twin 기초)
network = [
    Hospital(1, "Hanul Regional Center", ["CT", "thrombectomy", "neuro-ICU"], travel_time=18, confirm_delay=5, accept_prob=0.85, load=2, uncertainty=1),
    Hospital(2, "Mirae Medical Center", ["CT", "neuro-ICU"], travel_time=12, confirm_delay=8, accept_prob=0.60, load=8, uncertainty=3),
    Hospital(3, "Seobu University Hospital", ["CT", "thrombectomy", "neuro-ICU"], travel_time=27, confirm_delay=10, accept_prob=0.40, load=5, uncertainty=2),
    Hospital(4, "Local Clinic (No Neuro)", ["CT"], travel_time=5, confirm_delay=2, accept_prob=0.95, load=1, uncertainty=1)
]

# 3. 환자 발생: 뇌졸중 의심 환자 (CT 및 혈전제거술 필요)
required_caps = ["CT", "thrombectomy"]

# 4. 알고리즘 실행
print("🚨 SYNAPSE-H: RECOMMENDED DESTINATIONS 🚨")
print("-" * 50)
result = synapse_engine.rank_destinations(required_caps, network)
print(result)

🚨 SYNAPSE-H: RECOMMENDED DESTINATIONS 🚨
--------------------------------------------------
                    Hospital  Cost_Score Accept_Prob  ETA
0      Hanul Regional Center        38.5       85.0%  18m
1  Seobu University Hospital        92.5       40.0%  27m


In [4]:
import math
import copy
import json

# ==========================================
# [STEP 1] 자연어(NLP) Intake 모듈 
# ==========================================
class NLP_Intake_Module:
    def parse_multiple_notes(self, raw_notes):
        """
        여러 명의 비정형 의뢰서(텍스트)를 입력받아 구조화된 환자 객체 리스트로 반환합니다.
        (실제로는 LLM API가 각 텍스트를 분석하지만, 베타 데모를 위해 Mock-up 응답을 매칭합니다)
        """
        parsed_patients = []
        for i, text in enumerate(raw_notes):
            # LLM이 텍스트를 읽고 필요한 역량(Required Caps)을 추출했다고 가정
            if "facial droop" in text.lower():
                condition = "Stroke (Needs Thrombectomy)"
                caps = ["CT", "thrombectomy"]
            elif "mild head" in text.lower():
                condition = "Mild Head Injury"
                caps = ["CT"]
            else:
                condition = "Severe Trauma"
                caps = ["CT", "trauma"]
                
            parsed_patients.append({
                "p_id": f"P-00{i+1}",
                "condition": condition,
                "required_caps": caps,
                "confidence": 0.95
            })
        return parsed_patients

# ==========================================
# [STEP 2] ML 수용 확률 예측 모듈 & 데이터 구조
# ==========================================
class Patient:
    def __init__(self, p_id, condition, required_caps):
        self.p_id = p_id
        self.condition = condition
        self.required_caps = required_caps

class Hospital:
    def __init__(self, h_id, name, capabilities, capacity, travel_time, hist_rate, current_load, feed_freshness):
        self.h_id = h_id
        self.name = name
        self.capabilities = set(capabilities)
        self.capacity = capacity              # 남은 병상 수 (Resource)
        self.travel_time = travel_time
        
        # ML 입력 변수들
        self.hist_rate = hist_rate
        self.current_load = current_load
        self.feed_freshness = feed_freshness
        self.uncertainty = 1 if feed_freshness < 5 else 3
        
        # Logistic Regression 기반 수용 확률 예측 (ML)
        self.accept_prob = self.predict_prob()

    def predict_prob(self):
        # 과거 수용률, 혼잡도, 데이터 최신성을 종합해 0~1 사이 확률 계산
        z = 1.0 + (4.0 * self.hist_rate) - (0.5 * self.current_load) - (0.2 * self.feed_freshness)
        return 1 / (1 + math.exp(-z))

# ==========================================
# [STEP 3] 다중 환자 동시 최적화 스케줄링 (Linear Programming / DFS)
# ==========================================
class SynapseH_GlobalOptimizer:
    def __init__(self, weights):
        self.weights = weights # w1 ~ w5

    def calc_cost(self, hospital):
        w1, w2, w3, w4, w5 = self.weights
        cost = (w1 * hospital.travel_time +
                w2 * 5.0 + 
                w3 * (1 - hospital.accept_prob) * 100 + 
                w4 * hospital.current_load +
                w5 * hospital.uncertainty)
        return cost

    def solve_allocation(self, patients, hospitals):
        best_cost = float('inf')
        best_assignment = None

        def backtrack(patient_idx, current_total_cost, current_assignment, current_capacities):
            nonlocal best_cost, best_assignment
            
            if patient_idx == len(patients):
                if current_total_cost < best_cost:
                    best_cost = current_total_cost
                    best_assignment = copy.deepcopy(current_assignment)
                return

            p = patients[patient_idx]
            assigned_flag = False
            
            for h_idx, h in enumerate(hospitals):
                # 역량이 맞고(NLP 데이터), 병상이 남아있는 경우에만 배정
                if current_capacities[h_idx] > 0 and set(p.required_caps).issubset(h.capabilities):
                    cost = self.calc_cost(h)
                    
                    current_capacities[h_idx] -= 1
                    current_assignment.append((p.p_id, p.condition, h.name, cost))
                    
                    backtrack(patient_idx + 1, current_total_cost + cost, current_assignment, current_capacities)
                    
                    current_assignment.pop()
                    current_capacities[h_idx] += 1
                    assigned_flag = True
            
            if not assigned_flag:
                current_assignment.append((p.p_id, p.condition, "❌ ESCALATE (No Bed/Cap)", float('inf')))
                backtrack(patient_idx + 1, float('inf'), current_assignment, current_capacities)
                current_assignment.pop()

        initial_capacities = [h.capacity for h in hospitals]
        backtrack(0, 0.0, [], initial_capacities)

        return best_assignment, best_cost


# ==========================================
# 🚀 [최종 실행] SYNAPSE-H 시스템 통합 테스트
# ==========================================
if __name__ == "__main__":
    
    print("🚨 [SYNAPSE-H SYSTEM INITIATING...] 🚨\n")
    
    # 1. 텍스트 의뢰서 접수 (응급실 간호사가 다급하게 친 채팅이나 메모)
    raw_notes = [
        "72yo male, facial droop, left arm weakness. BP 160/90. Need immediate CT and mechanical thrombectomy.",
        "25yo female, mild head injury from fall. Stable but needs CT scan to rule out bleeding.",
        "40yo male, severe multi-trauma from car crash. Intubated. Needs trauma surgery and CT."
    ]
    
    print("📥 [STEP 1] 비정형 텍스트 수신 완료. NLP 파싱 시작...")
    nlp_module = NLP_Intake_Module()
    parsed_json_list = nlp_module.parse_multiple_notes(raw_notes)
    
    # NLP 결과를 Patient 객체로 변환
    patients = []
    for data in parsed_json_list:
        patients.append(Patient(data["p_id"], data["condition"], data["required_caps"]))
        print(f"   ✅ {data['p_id']} 파싱 완료: 필요 역량 {data['required_caps']}")
    print("")

    # 2. 지역 병원 네트워크 현황 가동 (ML 모듈이 실시간 수용 확률 계산)
    print("🧠 [STEP 2] 지역 병원망 로드 및 ML 수용 확률 예측 (데이터 최신성 반영)...")
    network = [
        Hospital(1, "Hanul Regional", ["CT", "thrombectomy", "neuro-ICU", "trauma"], capacity=1, travel_time=10, hist_rate=0.9, current_load=2, feed_freshness=2),
        Hospital(2, "Mirae Medical", ["CT", "neuro-ICU"], capacity=2, travel_time=15, hist_rate=0.7, current_load=5, feed_freshness=10),
        Hospital(3, "Seobu University", ["CT", "thrombectomy", "trauma"], capacity=2, travel_time=25, hist_rate=0.6, current_load=3, feed_freshness=5)
    ]
    for h in network:
        print(f"   🏥 {h.name}: 실시간 수용 확률 {int(h.accept_prob*100)}% 계산 완료")
    print("")

    # 3. 글로벌 최적화 스케줄링
    print("🛡️ [STEP 3] 다중 환자 동시 최적화(LP/Combinatorial) 스케줄링 가동...")
    optimizer = SynapseH_GlobalOptimizer(weights=(1.0, 1.2, 0.8, 0.5, 1.5))
    assignment, total_cost = optimizer.solve_allocation(patients, network)
    
    print("\n============================================================")
    print(f"🏆 최종 매칭 결과 (Total Network Cost: {round(total_cost, 2)}) 🏆")
    print("============================================================")
    for p_id, cond, h_name, cost in assignment:
        print(f"👤 환자 {p_id} [{cond}]")
        print(f"   ➡️ 배정 병원: 🏥 {h_name}")
        print("-" * 60)

🚨 [SYNAPSE-H SYSTEM INITIATING...] 🚨

📥 [STEP 1] 비정형 텍스트 수신 완료. NLP 파싱 시작...
   ✅ P-001 파싱 완료: 필요 역량 ['CT', 'thrombectomy']
   ✅ P-002 파싱 완료: 필요 역량 ['CT']
   ✅ P-003 파싱 완료: 필요 역량 ['CT', 'trauma']

🧠 [STEP 2] 지역 병원망 로드 및 ML 수용 확률 예측 (데이터 최신성 반영)...
   🏥 Hanul Regional: 실시간 수용 확률 96% 계산 완료
   🏥 Mirae Medical: 실시간 수용 확률 33% 계산 완료
   🏥 Seobu University: 실시간 수용 확률 71% 계산 완료

🛡️ [STEP 3] 다중 환자 동시 최적화(LP/Combinatorial) 스케줄링 가동...

🏆 최종 매칭 결과 (Total Network Cost: 141.88) 🏆
👤 환자 P-001 [Stroke (Needs Thrombectomy)]
   ➡️ 배정 병원: 🏥 Hanul Regional
------------------------------------------------------------
👤 환자 P-002 [Mild Head Injury]
   ➡️ 배정 병원: 🏥 Seobu University
------------------------------------------------------------
👤 환자 P-003 [Severe Trauma]
   ➡️ 배정 병원: 🏥 Seobu University
------------------------------------------------------------


In [ ]:
import streamlit as st
import pandas as pd

# 1. 시스템 백엔드

class Hospital:
    def __init__(self, id, name, capabilities, travel_time, confirm_delay, accept_prob, load, uncertainty):
        self.id = id
        self.name = name
        self.capabilities = set(capabilities) # 병원이 가진 역량
        
        # 최적화 계산 변수
        self.travel_time = travel_time        # 예상 이동 시간
        self.confirm_delay = confirm_delay    # 승인까지 예상 지연 시간
        self.accept_prob = accept_prob        # 수용 확률
        self.load = load                      # 혼잡도
        self.uncertainty = uncertainty        # 데이터 불확실성

class SynapseH_Optimizer:
    def __init__(self, weights):
        self.weights = weights # w1 ~ w5

    def calculate_cost(self, hospital):
        """다중 제약 수학적 목적 함수 (Cost 계산)"""
        w1, w2, w3, w4, w5 = self.weights
        cost = (w1 * hospital.travel_time +
                w2 * hospital.confirm_delay +
                w3 * (1 - hospital.accept_prob) * 100 + 
                w4 * hospital.load +
                w5 * hospital.uncertainty)
        return cost

    def rank_destinations(self, required_capabilities, hospital_network):
        eligible_hospitals = []
        required_set = set(required_capabilities)
        
        # 1. Hard Constraints 
        for h in hospital_network:
            if required_set.issubset(h.capabilities):
                eligible_hospitals.append(h)
        
        if not eligible_hospitals:
            return None # 매칭 실패 
            
        # 2. Cost 최적화 및 정렬
        ranked_list = []
        for h in eligible_hospitals:
            cost = self.calculate_cost(h)
            ranked_list.append({
                '순위': 0,
                '병원명(Hospital)': h.name, 
                '최적화 점수(Cost)': round(cost, 2), 
                '수용 확률(Accept_Prob)': f"{int(h.accept_prob*100)}%", 
                '예상 시간(ETA)': f"{h.travel_time}분"
            })
            
        # Cost 오름차순 정렬
        ranked_list = sorted(ranked_list, key=lambda x: x['최적화 점수(Cost)'])
        
        # 순위 부여
        for idx, row in enumerate(ranked_list):
            row['순위'] = f"{idx + 1}위"
            
        return pd.DataFrame(ranked_list[:3])

def nlp_parse_note(text):
    """자연어(NLP) Intake 모듈 가상화"""
    text_lower = text.lower()
    if "facial droop" in text_lower or "stroke" in text_lower:
        return "뇌졸중 의심 (Stroke - Needs Thrombectomy)", ["CT", "thrombectomy"]
    elif "mild head" in text_lower:
        return "경증 두부 손상 (Mild Head Injury)", ["CT"]
    elif "trauma" in text_lower or "crash" in text_lower:
        return "중증 다발성 외상 (Severe Trauma)", ["CT", "trauma"]
    else:
        return "기타 응급 질환 (General Emergency)", ["CT"]


# 2. 프론트엔드 UI (Streamlit)

st.set_page_config(page_title="SYNAPSE-H", layout="wide", initial_sidebar_state="expanded")

st.title("🚨 SYNAPSE-H | 응급 환자 이송 최적화 플랫폼")
st.markdown("의료진의 자연어 메모를 분석하여 가장 최적화된 이송 병원을 추천하는 의사결정 지원 시스템(MVP)입니다.")

# 가상의 지역 병원 네트워크 데이터 로드
network = [
    Hospital(1, "Hanul Regional Center", ["CT", "thrombectomy", "neuro-ICU", "trauma"], 18, 5, 0.85, 2, 1),
    Hospital(2, "Mirae Medical Center", ["CT", "neuro-ICU"], 12, 8, 0.60, 8, 3),
    Hospital(3, "Seobu University Hospital", ["CT", "thrombectomy", "neuro-ICU", "trauma"], 27, 10, 0.40, 5, 2),
    Hospital(4, "Local Clinic (No Neuro)", ["CT"], 5, 2, 0.95, 1, 1)
]

# 좌측 사이드바: 최적화 가중치 사용자 컨트롤
st.sidebar.header("⚖️ 최적화 가중치 제어 패널")
st.sidebar.caption("수학적 목적 함수(Objective Function)의 변수를 조정해 알고리즘 결과를 테스트해보세요.")
w1 = st.sidebar.slider("이동 시간 (Travel Time, T)", 0.0, 2.0, 1.0, 0.1)
w2 = st.sidebar.slider("승인 지연 (Confirm Delay, C)", 0.0, 2.0, 1.2, 0.1)
w3 = st.sidebar.slider("수용 거절 리스크 (Rejection Risk, A)", 0.0, 2.0, 0.8, 0.1)
w4 = st.sidebar.slider("병원 혼잡도 (Load, L)", 0.0, 2.0, 0.5, 0.1)
w5 = st.sidebar.slider("데이터 불확실성 (Uncertainty, U)", 0.0, 2.0, 1.5, 0.1)

# 메인 화면 영역 분할
col1, col2 = st.columns([1, 1])

with col1:
    st.subheader("1. 환자 의뢰서(Referral Note) 접수")
    st.info("실제 의료 현장처럼 텍스트를 입력하면 NLP 모듈이 필요 역량을 자동 파싱합니다.")
    
    sample_text = "72yo male, facial droop, left arm weakness. BP 160/90. Need immediate CT and mechanical thrombectomy."
    note_input = st.text_area("비정형 텍스트 입력창:", value=sample_text, height=150)
    
    run_button = st.button("🚀 SYNAPSE-H 매칭 알고리즘 실행", type="primary", use_container_width=True)

with col2:
    st.subheader("2. 지능형 매칭 결과 (Decision Layer)")
    
    if run_button:
        with st.spinner("AI가 비정형 데이터를 구조화하고 최적의 병원을 연산 중입니다..."):
            # NLP 파싱 결과
            condition, req_caps = nlp_parse_note(note_input)
            st.success(f"**[파싱 완료]** 예상 질환: {condition}")
            st.write(f"**필수 요구 역량 (Hard Constraints):** `{', '.join(req_caps)}`")
            
            # 엔진 구동 및 결과 도출
            optimizer = SynapseH_Optimizer((w1, w2, w3, w4, w5))
            result_df = optimizer.rank_destinations(req_caps, network)
            
            if result_df is not None:
                # 결과 테이블 출력
                st.dataframe(result_df, hide_index=True, use_container_width=True)
                
                # Top 1 추천 브리핑
                best_hospital = result_df.iloc[0]
                st.markdown(f"""
                <div style="background-color:#EBF5FB; padding:15px; border-radius:10px;">
                    <h4 style="margin-top:0; color:#154360;">✅ 최종 의료진(Human) 의사결정 대기</h4>
                    SYNAPSE-H는 현재 상황에서 최적의 이송 병원으로 <b>'{best_hospital['병원명(Hospital)']}'</b>을(를) 추천합니다.<br>
                    (예상 소요 시간: {best_hospital['예상 시간(ETA)']} / 수용 확률: {best_hospital['수용 확률(Accept_Prob)']})<br><br>
                    <b>이 병원으로 전원을 최종 승인하시겠습니까?</b>
                </div>
                """, unsafe_allow_html=True)
            else:
                # 안전 장치 작동 (조건에 맞는 병원 없음)
                st.error("🚨 ESCALATE TO MEDICAL CONTROL 🚨\n\n현재 지역 네트워크 내에 입력하신 필수 요구 역량을 만족하는 병원이 없습니다. 인간 의료 통제관(Medical Control)의 개입이 필요합니다.")
    else:
        st.write("의뢰서를 입력하고 실행 버튼을 눌러주세요.")